# 05 - Storytelling dos Resultados

Este notebook transforma os outputs finais da Aurora em uma narrativa curta para apresenta??o. Ele conecta dados, modelo, Power BI e impacto de neg?cio.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DADOS_RAW = ROOT / "dados" / "raw"
DADOS_PROCESSED = ROOT / "dados" / "processed"
DADOS_OUTPUTS = ROOT / "dados" / "outputs"
print(f"Raiz do projeto: {ROOT}")

import json

In [ ]:
with open(DADOS_OUTPUTS / "metricas_modelo.json", encoding="utf-8") as f:
    metricas = json.load(f)

predicoes = pd.read_csv(DADOS_OUTPUTS / "predicoes_churn.csv")
feature_importance = pd.read_csv(DADOS_OUTPUTS / "feature_importance.csv")
threshold_analysis = pd.read_csv(DADOS_OUTPUTS / "threshold_analysis.csv")
summary_path = DADOS_OUTPUTS / "summary.json"
summary = json.load(open(summary_path, encoding="utf-8")) if summary_path.exists() else {}

print("Arquivos carregados com sucesso.")

## 1. Resumo executivo do modelo

As m?tricas s?o geradas dinamicamente pelo pipeline. O README n?o fixa valores para evitar inconsist?ncia entre execu??es.

In [ ]:
metricas_principais = {k: metricas.get(k) for k in ["accuracy", "precision", "recall", "f1_score", "roc_auc", "baseline_churn_rate", "threshold_used"]}
display(pd.DataFrame(metricas_principais.items(), columns=["metrica", "valor"]))
print(metricas.get("recommended_threshold_reason", ""))

**Leitura para apresenta??o:** ROC-AUC mostra capacidade de ranking, precision mostra qualidade dos alertas e recall mostra capacidade de capturar clientes que poderiam sair.

## 2. Clientes priorit?rios para reten??o

In [ ]:
cols = ["cliente_id", "nome", "estado", "perfil_risco", "prob_churn", "risco", "recomendacao"]
display(predicoes[cols].head(20))

risco = predicoes["risco"].value_counts().rename_axis("risco").reset_index(name="clientes")
display(risco)

**Leitura para apresenta??o:** a Aurora n?o decide automaticamente. Ela organiza uma fila de prioriza??o para que a equipe humana de reten??o aja com contexto.

## 3. Principais vari?veis do modelo

In [ ]:
display(feature_importance.head(15))
feature_importance.head(10).sort_values("importance").plot(kind="barh", x="feature", y="importance", figsize=(8, 5), color="#8558f2", legend=False)
plt.title("Sinais mais importantes do modelo")
plt.tight_layout()
plt.show()

**Leitura para apresenta??o:** feature importance ajuda a explicar quais sinais sustentam o ranking de risco, conectando modelo e decis?o de neg?cio.

## 4. Threshold analysis

A Aurora compara thresholds para reten??o. Em churn, reduzir falso negativo costuma ser importante, mas sem criar alertas demais para a equipe operacional.

In [ ]:
display(threshold_analysis)

plt.figure(figsize=(7, 4))
plt.plot(threshold_analysis["threshold"], threshold_analysis["precision"], marker="o", label="precision")
plt.plot(threshold_analysis["threshold"], threshold_analysis["recall"], marker="o", label="recall")
plt.plot(threshold_analysis["threshold"], threshold_analysis["f1_score"], marker="o", label="f1_score")
plt.title("Compara??o de thresholds")
plt.xlabel("Threshold")
plt.ylabel("M?trica")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Como conectar ao Power BI

Na apresenta??o, mostre as p?ginas nesta ordem:

1. **Vis?o Executiva:** KPIs principais, volume financeiro e distribui??o de risco.
2. **Consumo e Comportamento Financeiro:** categorias, canais e matriz de perfil de risco.
3. **Churn e Reten??o:** clientes priorit?rios e recomenda??es.
4. **Modelo ML:** m?tricas, feature importance e threshold analysis.
5. **Storytelling Executivo:** s?ntese do problema, solu??o, impacto e pr?ximos passos.

In [ ]:
print("Resumo do dataset para fala:")
print("Clientes:", summary.get("total_clientes", "N/D"))
print("Transa??es:", summary.get("total_transacoes", "N/D"))
print("Fonte:", summary.get("data_source_name", "N/D"))
print("Transa??es sint?ticas usadas:", summary.get("synthetic_transactions_used", "N/D"))

## 6. Roteiro curto para 5 minutos

**Problema:** empresas financeiras percebem tarde demais sinais de churn.

**Dados:** usamos o dataset p?blico Churn Modelling do Kaggle como base de clientes e churn. Como ele n?o possui hist?rico transacional, criamos uma camada sint?tica reprodut?vel de transa??es.

**An?lise:** exploramos renda, saldo, score, perfil de risco, estado, categorias e evolu??o mensal.

**Modelo:** treinamos Random Forest com class weight balanced, avaliando precision, recall, F1, ROC-AUC e thresholds.

**Dashboard:** o Power BI mostra vis?o executiva, consumo, churn, reten??o e explica??o do modelo.

**Impacto:** a Aurora prioriza clientes em risco para reten??o consultiva, apoiando decis?o humana com dados.

**B?nus:** o app React premium e a An?lise Expressa mostram como a solu??o poderia virar produto est?tico, sem backend obrigat?rio.